# HW10-11 — Компьютерное зрение: CNN, transfer learning, сегментация

- **Часть A (S10):** классификация CIFAR-100 — простая CNN, аугментации, ResNet18 transfer learning (C1–C4)
- **Часть B (S11):** сегментация OxfordIIITPet — pretrained DeepLabV3, два режима постобработки (V1–V2)

## 2.4.1 Импорты, seed и устройство

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset
import torchvision
import torchvision.transforms as transforms
from torchvision import models
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import csv
import json
import copy
import os
import ssl

ssl._create_default_https_context = ssl._create_unverified_context

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version: {torch.__version__}')
print(f'Device: {DEVICE}')

PyTorch version: 2.5.1+cu121
Device: cpu


## 2.4.2 Данные и DataLoader — Часть A (CIFAR-100)

In [3]:
CIFAR_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR_STD  = (0.2675, 0.2565, 0.2761)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
NUM_CLASSES = 100

transform_base = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

transform_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

transform_resnet_train = transforms.Compose([
    transforms.Resize(64),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

transform_resnet_eval = transforms.Compose([
    transforms.Resize(64),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

full_train = torchvision.datasets.CIFAR100(
    root='./data', train=True, download=True, transform=transform_base
)
test_dataset_base = torchvision.datasets.CIFAR100(
    root='./data', train=False, download=True, transform=transform_base
)

SUBSET_SIZE = 8000

full_idx = torch.randperm(len(full_train), generator=torch.Generator().manual_seed(SEED)).tolist()
subset_idx = full_idx[:SUBSET_SIZE]

train_size = int(0.8 * SUBSET_SIZE)
val_size = SUBSET_SIZE - train_size
train_idx = subset_idx[:train_size]
val_idx = subset_idx[train_size:]

print(f'CIFAR-100: {NUM_CLASSES} classes')
print(f'Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_dataset_base)}')
print(f'(Subset {SUBSET_SIZE} из {len(full_train)} для ускорения на CPU)')

Files already downloaded and verified


c:\Users\igork\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Files already downloaded and verified
CIFAR-100: 100 classes
Train: 6400, Val: 1600, Test: 10000
(Subset 8000 из 50000 для ускорения на CPU)


In [4]:
def make_loaders(train_transform, eval_transform, batch_size, train_idx=train_idx, val_idx=val_idx):
    """Create train/val/test loaders with given transforms."""
    tr = torchvision.datasets.CIFAR100(root='./data', train=True, download=False, transform=train_transform)
    ev = torchvision.datasets.CIFAR100(root='./data', train=True, download=False, transform=eval_transform)
    te = torchvision.datasets.CIFAR100(root='./data', train=False, download=False, transform=eval_transform)
    train_loader = DataLoader(Subset(tr, train_idx), batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(Subset(ev, val_idx),   batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(te, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader

### Sanity-check и примеры изображений

In [5]:
train_loader_base, val_loader_base, test_loader_base = make_loaders(
    transform_base, transform_base, batch_size=128
)

x_batch, y_batch = next(iter(train_loader_base))
print(f'x_batch.shape: {x_batch.shape}')
print(f'y_batch.shape: {y_batch.shape}')
print(f'x range: [{x_batch.min():.2f}, {x_batch.max():.2f}]')
print(f'y unique (first 20): {sorted(y_batch.unique().tolist())[:20]}...')

raw_dataset = torchvision.datasets.CIFAR100(root='./data', train=True, download=False, transform=transforms.ToTensor())
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img, label = raw_dataset[train_idx[i]]
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(f'class {label}')
    ax.axis('off')
plt.suptitle('CIFAR-100: примеры изображений')
plt.tight_layout()
plt.show()

x_batch.shape: torch.Size([128, 3, 32, 32])
y_batch.shape: torch.Size([128])
x range: [-1.90, 2.03]
y unique (first 20): [1, 2, 3, 4, 5, 8, 9, 10, 11, 12, 13, 14, 15, 16, 18, 19, 20, 22, 23, 24]...


C:\Users\igork\AppData\Local\Temp\ipykernel_9992\2113090794.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Визуализация аугментаций → `augmentations_preview.png`

In [6]:
raw_pil = torchvision.datasets.CIFAR100(root='./data', train=True, download=False)

aug_transform_vis = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
])

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for col in range(5):
    pil_img, label = raw_pil[train_idx[col]]
    axes[0, col].imshow(pil_img)
    axes[0, col].set_title(f'Original (cls {label})')
    axes[0, col].axis('off')

    aug_img = aug_transform_vis(pil_img)
    axes[1, col].imshow(aug_img.permute(1, 2, 0).clamp(0, 1).numpy())
    axes[1, col].set_title('Augmented')
    axes[1, col].axis('off')

plt.suptitle('Аугментации CIFAR-100')
plt.tight_layout()
os.makedirs('artifacts/figures', exist_ok=True)
fig.savefig('artifacts/figures/augmentations_preview.png', dpi=150, bbox_inches='tight')
print('Saved: artifacts/figures/augmentations_preview.png')
plt.show()

Saved: artifacts/figures/augmentations_preview.png


C:\Users\igork\AppData\Local\Temp\ipykernel_9992\469609470.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2.4.3 Модели и функции обучения

In [7]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [8]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += x.size(0)
    return running_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            running_loss += loss.item() * x.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            total += x.size(0)
    return running_loss / total, correct / total


def run_experiment(model, train_loader, val_loader, criterion, optimizer,
                   device, num_epochs):
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0
    best_val_loss = float('inf')
    best_state = None

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f'  Epoch {epoch:02d} | '
              f'train_loss={train_loss:.4f}  train_acc={train_acc:.4f} | '
              f'val_loss={val_loss:.4f}  val_acc={val_acc:.4f}')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())

    epochs_trained = len(history['train_loss'])
    return history, best_val_acc, best_val_loss, best_state, epochs_trained

---
## Часть A (S10): классификация, аугментации и transfer learning (C1–C4)

### C1 — SimpleCNN без аугментаций

In [9]:
NUM_EPOCHS_CNN = 8
BATCH_SIZE_CNN = 256
LR_CNN = 1e-3

train_loader_c1, val_loader_c1, test_loader_c1 = make_loaders(
    transform_base, transform_base, BATCH_SIZE_CNN
)

torch.manual_seed(SEED)
model_c1 = SimpleCNN(NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer_c1 = optim.Adam(model_c1.parameters(), lr=LR_CNN)

print('=== C1: SimpleCNN (без аугментаций) ===')
hist_c1, best_acc_c1, best_loss_c1, state_c1, epochs_c1 = run_experiment(
    model_c1, train_loader_c1, val_loader_c1, criterion, optimizer_c1, DEVICE, NUM_EPOCHS_CNN
)
print(f'\nC1 best val_acc: {best_acc_c1:.4f}')

=== C1: SimpleCNN (без аугментаций) ===
  Epoch 01 | train_loss=4.4504  train_acc=0.0387 | val_loss=4.3678  val_acc=0.0344
  Epoch 02 | train_loss=4.1403  train_acc=0.0653 | val_loss=4.0257  val_acc=0.0644
  Epoch 03 | train_loss=3.8843  train_acc=0.0886 | val_loss=3.8423  val_acc=0.0919
  Epoch 04 | train_loss=3.7302  train_acc=0.1027 | val_loss=3.7695  val_acc=0.1119
  Epoch 05 | train_loss=3.5729  train_acc=0.1255 | val_loss=3.6692  val_acc=0.1150
  Epoch 06 | train_loss=3.4421  train_acc=0.1483 | val_loss=3.5791  val_acc=0.1219
  Epoch 07 | train_loss=3.3335  train_acc=0.1655 | val_loss=3.4282  val_acc=0.1481
  Epoch 08 | train_loss=3.2010  train_acc=0.1862 | val_loss=3.5318  val_acc=0.1444

C1 best val_acc: 0.1481


### C2 — SimpleCNN с аугментациями

In [10]:
train_loader_c2, val_loader_c2, test_loader_c2 = make_loaders(
    transform_aug, transform_base, BATCH_SIZE_CNN
)

torch.manual_seed(SEED)
model_c2 = SimpleCNN(NUM_CLASSES).to(DEVICE)
optimizer_c2 = optim.Adam(model_c2.parameters(), lr=LR_CNN)

print('=== C2: SimpleCNN (с аугментациями) ===')
hist_c2, best_acc_c2, best_loss_c2, state_c2, epochs_c2 = run_experiment(
    model_c2, train_loader_c2, val_loader_c2, criterion, optimizer_c2, DEVICE, NUM_EPOCHS_CNN
)
print(f'\nC2 best val_acc: {best_acc_c2:.4f}')

=== C2: SimpleCNN (с аугментациями) ===
  Epoch 01 | train_loss=4.4817  train_acc=0.0322 | val_loss=4.3645  val_acc=0.0306
  Epoch 02 | train_loss=4.2050  train_acc=0.0586 | val_loss=3.9901  val_acc=0.0794
  Epoch 03 | train_loss=3.9771  train_acc=0.0809 | val_loss=3.9263  val_acc=0.0788
  Epoch 04 | train_loss=3.8055  train_acc=0.0961 | val_loss=3.8984  val_acc=0.0969
  Epoch 05 | train_loss=3.6980  train_acc=0.1050 | val_loss=3.7358  val_acc=0.1119
  Epoch 06 | train_loss=3.5844  train_acc=0.1269 | val_loss=3.6515  val_acc=0.1412
  Epoch 07 | train_loss=3.4975  train_acc=0.1469 | val_loss=3.7435  val_acc=0.1119
  Epoch 08 | train_loss=3.4119  train_acc=0.1552 | val_loss=3.5788  val_acc=0.1331

C2 best val_acc: 0.1412


### C3 — ResNet18 head-only (backbone заморожен)

In [11]:
NUM_EPOCHS_RESNET = 6
BATCH_SIZE_RESNET = 128
LR_HEAD = 1e-3

train_loader_c3, val_loader_c3, test_loader_c3 = make_loaders(
    transform_resnet_train, transform_resnet_eval, BATCH_SIZE_RESNET
)

torch.manual_seed(SEED)
model_c3 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for param in model_c3.parameters():
    param.requires_grad = False
model_c3.fc = nn.Linear(model_c3.fc.in_features, NUM_CLASSES)
model_c3 = model_c3.to(DEVICE)

optimizer_c3 = optim.Adam(model_c3.fc.parameters(), lr=LR_HEAD)

print('=== C3: ResNet18 head-only ===')
hist_c3, best_acc_c3, best_loss_c3, state_c3, epochs_c3 = run_experiment(
    model_c3, train_loader_c3, val_loader_c3, criterion, optimizer_c3, DEVICE, NUM_EPOCHS_RESNET
)
print(f'\nC3 best val_acc: {best_acc_c3:.4f}')

=== C3: ResNet18 head-only ===
  Epoch 01 | train_loss=4.1488  train_acc=0.1150 | val_loss=3.4367  val_acc=0.2350
  Epoch 02 | train_loss=2.8894  train_acc=0.3470 | val_loss=2.9248  val_acc=0.3094
  Epoch 03 | train_loss=2.4062  train_acc=0.4336 | val_loss=2.7039  val_acc=0.3513
  Epoch 04 | train_loss=2.1342  train_acc=0.4877 | val_loss=2.6171  val_acc=0.3638
  Epoch 05 | train_loss=1.9530  train_acc=0.5161 | val_loss=2.5703  val_acc=0.3719
  Epoch 06 | train_loss=1.8065  train_acc=0.5450 | val_loss=2.5692  val_acc=0.3681

C3 best val_acc: 0.3719


### C4 — ResNet18 partial fine-tune (layer4 + fc)

In [12]:
LR_FINETUNE = 1e-4

torch.manual_seed(SEED)
model_c4 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for param in model_c4.parameters():
    param.requires_grad = False
for param in model_c4.layer4.parameters():
    param.requires_grad = True
model_c4.fc = nn.Linear(model_c4.fc.in_features, NUM_CLASSES)
model_c4 = model_c4.to(DEVICE)

optimizer_c4 = optim.Adam(
    filter(lambda p: p.requires_grad, model_c4.parameters()), lr=LR_FINETUNE
)

train_loader_c4, val_loader_c4, test_loader_c4 = make_loaders(
    transform_resnet_train, transform_resnet_eval, BATCH_SIZE_RESNET
)

print('=== C4: ResNet18 finetune (layer4 + fc) ===')
hist_c4, best_acc_c4, best_loss_c4, state_c4, epochs_c4 = run_experiment(
    model_c4, train_loader_c4, val_loader_c4, criterion, optimizer_c4, DEVICE, NUM_EPOCHS_RESNET
)
print(f'\nC4 best val_acc: {best_acc_c4:.4f}')

=== C4: ResNet18 finetune (layer4 + fc) ===
  Epoch 01 | train_loss=4.1490  train_acc=0.1206 | val_loss=3.3945  val_acc=0.2681
  Epoch 02 | train_loss=2.6253  train_acc=0.4544 | val_loss=2.7204  val_acc=0.3706
  Epoch 03 | train_loss=1.8807  train_acc=0.6209 | val_loss=2.3393  val_acc=0.4381
  Epoch 04 | train_loss=1.4021  train_acc=0.7247 | val_loss=2.1790  val_acc=0.4619
  Epoch 05 | train_loss=1.0461  train_acc=0.8192 | val_loss=2.0660  val_acc=0.4781
  Epoch 06 | train_loss=0.7708  train_acc=0.8800 | val_loss=1.9990  val_acc=0.4925

C4 best val_acc: 0.4925


### Выбор лучшей модели и финальная оценка на test

In [13]:
results_a = {
    'C1': (best_acc_c1, best_loss_c1, state_c1, hist_c1, epochs_c1),
    'C2': (best_acc_c2, best_loss_c2, state_c2, hist_c2, epochs_c2),
    'C3': (best_acc_c3, best_loss_c3, state_c3, hist_c3, epochs_c3),
    'C4': (best_acc_c4, best_loss_c4, state_c4, hist_c4, epochs_c4),
}

print('=== Сравнение C1–C4 ===')
for name, (acc, loss, *_) in results_a.items():
    print(f'  {name}: val_acc={acc:.4f}, val_loss={loss:.4f}')

best_id = max(results_a, key=lambda k: results_a[k][0])
best_acc_a, best_loss_a, best_state_a, best_hist_a, best_epochs_a = results_a[best_id]
print(f'\n=> Лучший: {best_id} (val_acc={best_acc_a:.4f})')

is_resnet_best = best_id in ('C3', 'C4')
if is_resnet_best:
    if best_id == 'C3':
        eval_model = models.resnet18(weights=None)
        eval_model.fc = nn.Linear(eval_model.fc.in_features, NUM_CLASSES)
    else:
        eval_model = models.resnet18(weights=None)
        eval_model.fc = nn.Linear(eval_model.fc.in_features, NUM_CLASSES)
    eval_model.load_state_dict(best_state_a)
    eval_model = eval_model.to(DEVICE)
    _, _, test_loader_best = make_loaders(
        transform_resnet_eval, transform_resnet_eval, BATCH_SIZE_RESNET
    )
else:
    eval_model = SimpleCNN(NUM_CLASSES).to(DEVICE)
    eval_model.load_state_dict(best_state_a)
    _, _, test_loader_best = make_loaders(
        transform_base, transform_base, BATCH_SIZE_CNN
    )

test_loss, test_acc = evaluate(eval_model, test_loader_best, criterion, DEVICE)
print(f'\n=== Финальная оценка на TEST ===')
print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_acc:.4f}')

=== Сравнение C1–C4 ===
  C1: val_acc=0.1481, val_loss=3.4282
  C2: val_acc=0.1412, val_loss=3.6515
  C3: val_acc=0.3719, val_loss=2.5703
  C4: val_acc=0.4925, val_loss=1.9990

=> Лучший: C4 (val_acc=0.4925)

=== Финальная оценка на TEST ===
Test loss: 2.0419
Test accuracy: 0.4852


### Графики части A

In [14]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs_range = range(1, len(best_hist_a['train_loss']) + 1)
axes[0].plot(epochs_range, best_hist_a['train_loss'], label='train_loss')
axes[0].plot(epochs_range, best_hist_a['val_loss'], label='val_loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title(f'{best_id}: Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, best_hist_a['train_acc'], label='train_acc')
axes[1].plot(epochs_range, best_hist_a['val_acc'], label='val_acc')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title(f'{best_id}: Accuracy'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig('artifacts/figures/classification_curves_best.png', dpi=150, bbox_inches='tight')
print('Saved: artifacts/figures/classification_curves_best.png')
plt.show()

Saved: artifacts/figures/classification_curves_best.png


C:\Users\igork\AppData\Local\Temp\ipykernel_9992\3030854630.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
names = ['C1', 'C2', 'C3', 'C4']
accs = [best_acc_c1, best_acc_c2, best_acc_c3, best_acc_c4]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(names, accs)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{acc:.4f}', ha='center', va='bottom', fontsize=11)
ax.set_ylabel('best_val_accuracy')
ax.set_title('Сравнение C1–C4: val accuracy')
ax.set_ylim(0, max(accs) * 1.15)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig('artifacts/figures/classification_compare.png', dpi=150, bbox_inches='tight')
print('Saved: artifacts/figures/classification_compare.png')
plt.show()

Saved: artifacts/figures/classification_compare.png


C:\Users\igork\AppData\Local\Temp\ipykernel_9992\745070533.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Сохранение артефактов части A

In [16]:
torch.save(best_state_a, 'artifacts/best_classifier.pt')
print('Saved: artifacts/best_classifier.pt')

if is_resnet_best:
    model_summary_best = f'ResNet18 ({"head-only" if best_id=="C3" else "finetune layer4+fc"})'
    lr_best = LR_HEAD if best_id == 'C3' else LR_FINETUNE
    bs_best = BATCH_SIZE_RESNET
else:
    model_summary_best = 'SimpleCNN(3 conv blocks + FC)'
    lr_best = LR_CNN
    bs_best = BATCH_SIZE_CNN

config = {
    'dataset': 'CIFAR100',
    'seed': SEED,
    'num_classes': NUM_CLASSES,
    'model': model_summary_best,
    'experiment_id': best_id,
    'optimizer': 'Adam',
    'lr': lr_best,
    'batch_size': bs_best,
    'epochs_trained': best_epochs_a,
    'best_val_accuracy': round(best_acc_a, 4),
    'test_accuracy': round(test_acc, 4),
}
with open('artifacts/best_classifier_config.json', 'w') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
print('Saved: artifacts/best_classifier_config.json')

Saved: artifacts/best_classifier.pt
Saved: artifacts/best_classifier_config.json


---
## Часть B (S11): сегментация OxfordIIITPet (V1–V2)

### Загрузка данных и pretrained модели

In [17]:
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights
from PIL import Image
import scipy.ndimage as ndimage

pet_dataset = torchvision.datasets.OxfordIIITPet(
    root='./data', split='test', target_types='segmentation', download=True
)
print(f'OxfordIIITPet test: {len(pet_dataset)} images')

seg_model = deeplabv3_resnet50(weights=DeepLabV3_ResNet50_Weights.DEFAULT).to(DEVICE)
seg_model.eval()
print('DeepLabV3_ResNet50 loaded (pretrained on PASCAL VOC, 21 classes)')

OxfordIIITPet test: 3669 images
DeepLabV3_ResNet50 loaded (pretrained on PASCAL VOC, 21 classes)


In [18]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i in range(4):
    img, mask = pet_dataset[i * 50]
    axes[0, i].imshow(img)
    axes[0, i].set_title(f'Image {i * 50}'); axes[0, i].axis('off')
    mask_np = np.array(mask)
    axes[1, i].imshow(mask_np, cmap='tab10', vmin=1, vmax=3)
    axes[1, i].set_title('GT mask (1=fg, 2=border, 3=bg)'); axes[1, i].axis('off')
plt.suptitle('OxfordIIITPet: примеры')
plt.tight_layout()
plt.show()

C:\Users\igork\AppData\Local\Temp\ipykernel_9992\3314930700.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Функции для inference и метрик

In [19]:
seg_preprocess = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

ANIMAL_CLASSES = [3, 8, 10, 12, 13, 15, 17, 18, 19, 20]

def predict_mask(model, pil_img, device, postprocess='basic'):
    """Run DeepLabV3 and return binary foreground mask."""
    w, h = pil_img.size
    inp = seg_preprocess(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(inp)['out'][0]
    probs = torch.softmax(out, dim=0)
    fg_prob = probs[ANIMAL_CLASSES].sum(dim=0).cpu().numpy()

    if postprocess == 'basic':
        binary = (fg_prob > 0.5).astype(np.uint8)
    elif postprocess == 'morpho':
        binary = (fg_prob > 0.4).astype(np.uint8)
        binary = ndimage.binary_closing(binary, structure=np.ones((5, 5)), iterations=2).astype(np.uint8)
        binary = ndimage.binary_opening(binary, structure=np.ones((3, 3)), iterations=1).astype(np.uint8)
    else:
        raise ValueError(f'Unknown postprocess: {postprocess}')

    mask_resized = np.array(Image.fromarray(binary).resize((w, h), Image.NEAREST))
    return mask_resized


def compute_seg_metrics(pred, gt_fg):
    """Compute IoU, precision, recall for binary foreground masks."""
    pred = pred.astype(bool)
    gt = gt_fg.astype(bool)
    intersection = (pred & gt).sum()
    union = (pred | gt).sum()
    iou = intersection / union if union > 0 else 0.0
    precision = intersection / pred.sum() if pred.sum() > 0 else 0.0
    recall = intersection / gt.sum() if gt.sum() > 0 else 0.0
    return float(iou), float(precision), float(recall)

### V1 — базовая постобработка (порог 0.5)

In [20]:
N_EVAL = min(200, len(pet_dataset))
eval_indices = list(range(N_EVAL))

ious_v1, precs_v1, recs_v1 = [], [], []
for idx in eval_indices:
    pil_img, gt_mask_pil = pet_dataset[idx]
    gt_mask = np.array(gt_mask_pil)
    gt_fg = (gt_mask == 1).astype(np.uint8)

    pred = predict_mask(seg_model, pil_img, DEVICE, postprocess='basic')
    iou, prec, rec = compute_seg_metrics(pred, gt_fg)
    ious_v1.append(iou); precs_v1.append(prec); recs_v1.append(rec)

mean_iou_v1 = np.mean(ious_v1)
mean_prec_v1 = np.mean(precs_v1)
mean_rec_v1 = np.mean(recs_v1)
print(f'=== V1 (basic, threshold=0.5) ===')
print(f'  mean_iou:        {mean_iou_v1:.4f}')
print(f'  pixel_precision: {mean_prec_v1:.4f}')
print(f'  pixel_recall:    {mean_rec_v1:.4f}')

=== V1 (basic, threshold=0.5) ===
  mean_iou:        0.6893
  pixel_precision: 0.6975
  pixel_recall:    0.9857


### V2 — морфологическая постобработка

In [21]:
ious_v2, precs_v2, recs_v2 = [], [], []
for idx in eval_indices:
    pil_img, gt_mask_pil = pet_dataset[idx]
    gt_mask = np.array(gt_mask_pil)
    gt_fg = (gt_mask == 1).astype(np.uint8)

    pred = predict_mask(seg_model, pil_img, DEVICE, postprocess='morpho')
    iou, prec, rec = compute_seg_metrics(pred, gt_fg)
    ious_v2.append(iou); precs_v2.append(prec); recs_v2.append(rec)

mean_iou_v2 = np.mean(ious_v2)
mean_prec_v2 = np.mean(precs_v2)
mean_rec_v2 = np.mean(recs_v2)
print(f'=== V2 (morpho, threshold=0.4 + close/open) ===')
print(f'  mean_iou:        {mean_iou_v2:.4f}')
print(f'  pixel_precision: {mean_prec_v2:.4f}')
print(f'  pixel_recall:    {mean_rec_v2:.4f}')

=== V2 (morpho, threshold=0.4 + close/open) ===
  mean_iou:        0.6639
  pixel_precision: 0.6746
  pixel_recall:    0.9815


### Визуализация сегментации → `segmentation_examples.png`

In [22]:
vis_indices = [0, 25, 50, 75]
fig, axes = plt.subplots(len(vis_indices), 4, figsize=(16, 4 * len(vis_indices)))

for row, idx in enumerate(vis_indices):
    pil_img, gt_mask_pil = pet_dataset[idx]
    gt_mask = np.array(gt_mask_pil)
    gt_fg = (gt_mask == 1).astype(np.uint8)
    pred_v1 = predict_mask(seg_model, pil_img, DEVICE, postprocess='basic')
    pred_v2 = predict_mask(seg_model, pil_img, DEVICE, postprocess='morpho')

    axes[row, 0].imshow(pil_img); axes[row, 0].set_title('Original'); axes[row, 0].axis('off')
    axes[row, 1].imshow(gt_fg, cmap='gray'); axes[row, 1].set_title('GT foreground'); axes[row, 1].axis('off')
    axes[row, 2].imshow(pred_v1, cmap='gray'); axes[row, 2].set_title(f'V1 (IoU={ious_v1[idx]:.3f})'); axes[row, 2].axis('off')
    axes[row, 3].imshow(pred_v2, cmap='gray'); axes[row, 3].set_title(f'V2 (IoU={ious_v2[idx]:.3f})'); axes[row, 3].axis('off')

plt.suptitle('Сегментация OxfordIIITPet: V1 vs V2')
plt.tight_layout()
fig.savefig('artifacts/figures/segmentation_examples.png', dpi=150, bbox_inches='tight')
print('Saved: artifacts/figures/segmentation_examples.png')
plt.show()

Saved: artifacts/figures/segmentation_examples.png


C:\Users\igork\AppData\Local\Temp\ipykernel_9992\4068681400.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Сравнение метрик V1 vs V2 → `segmentation_metrics.png`

In [23]:
metrics = ['mean_iou', 'pixel_precision', 'pixel_recall']
v1_vals = [mean_iou_v1, mean_prec_v1, mean_rec_v1]
v2_vals = [mean_iou_v2, mean_prec_v2, mean_rec_v2]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, v1_vals, width, label='V1 (basic)')
bars2 = ax.bar(x + width/2, v2_vals, width, label='V2 (morpho)')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylabel('Score')
ax.set_title('Segmentation: V1 vs V2')
ax.legend()
ax.set_ylim(0, max(max(v1_vals), max(v2_vals)) * 1.15)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig('artifacts/figures/segmentation_metrics.png', dpi=150, bbox_inches='tight')
print('Saved: artifacts/figures/segmentation_metrics.png')
plt.show()

Saved: artifacts/figures/segmentation_metrics.png


C:\Users\igork\AppData\Local\Temp\ipykernel_9992\3787756972.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Сохранение runs.csv

In [24]:
fieldnames = [
    'experiment_id', 'task', 'dataset', 'seed', 'model_summary',
    'optimizer', 'lr', 'epochs_trained',
    'best_val_accuracy', 'test_accuracy',
    'precision', 'recall', 'mean_iou', 'notes'
]

rows = [
    {'experiment_id': 'C1', 'task': 'classification', 'dataset': 'CIFAR100',
     'seed': SEED, 'model_summary': 'SimpleCNN(3 conv blocks)/no_aug',
     'optimizer': 'Adam', 'lr': LR_CNN, 'epochs_trained': epochs_c1,
     'best_val_accuracy': round(best_acc_c1, 4), 'test_accuracy': '',
     'precision': '', 'recall': '', 'mean_iou': '', 'notes': 'baseline CNN'},

    {'experiment_id': 'C2', 'task': 'classification', 'dataset': 'CIFAR100',
     'seed': SEED, 'model_summary': 'SimpleCNN(3 conv blocks)/aug',
     'optimizer': 'Adam', 'lr': LR_CNN, 'epochs_trained': epochs_c2,
     'best_val_accuracy': round(best_acc_c2, 4), 'test_accuracy': '',
     'precision': '', 'recall': '', 'mean_iou': '', 'notes': 'with augmentations'},

    {'experiment_id': 'C3', 'task': 'classification', 'dataset': 'CIFAR100',
     'seed': SEED, 'model_summary': 'ResNet18/head-only',
     'optimizer': 'Adam', 'lr': LR_HEAD, 'epochs_trained': epochs_c3,
     'best_val_accuracy': round(best_acc_c3, 4), 'test_accuracy': '',
     'precision': '', 'recall': '', 'mean_iou': '', 'notes': 'backbone frozen'},

    {'experiment_id': 'C4', 'task': 'classification', 'dataset': 'CIFAR100',
     'seed': SEED, 'model_summary': 'ResNet18/finetune(layer4+fc)',
     'optimizer': 'Adam', 'lr': LR_FINETUNE, 'epochs_trained': epochs_c4,
     'best_val_accuracy': round(best_acc_c4, 4), 'test_accuracy': '',
     'precision': '', 'recall': '', 'mean_iou': '', 'notes': 'partial finetune'},

    {'experiment_id': 'V1', 'task': 'segmentation', 'dataset': 'OxfordIIITPet',
     'seed': SEED, 'model_summary': 'DeepLabV3_ResNet50/pretrained',
     'optimizer': '', 'lr': '', 'epochs_trained': 0,
     'best_val_accuracy': '', 'test_accuracy': '',
     'precision': round(mean_prec_v1, 4), 'recall': round(mean_rec_v1, 4),
     'mean_iou': round(mean_iou_v1, 4), 'notes': 'basic postprocess (threshold=0.5)'},

    {'experiment_id': 'V2', 'task': 'segmentation', 'dataset': 'OxfordIIITPet',
     'seed': SEED, 'model_summary': 'DeepLabV3_ResNet50/pretrained',
     'optimizer': '', 'lr': '', 'epochs_trained': 0,
     'best_val_accuracy': '', 'test_accuracy': '',
     'precision': round(mean_prec_v2, 4), 'recall': round(mean_rec_v2, 4),
     'mean_iou': round(mean_iou_v2, 4), 'notes': 'morpho postprocess (threshold=0.4 + close/open)'},
]

best_row_idx = {'C1': 0, 'C2': 1, 'C3': 2, 'C4': 3}[best_id]
rows[best_row_idx]['test_accuracy'] = round(test_acc, 4)

with open('artifacts/runs.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)
print('Saved: artifacts/runs.csv')

print('\n=== Сводка ===')
print(f'{"ID":<5} {"task":<16} {"val_acc":<10} {"test_acc":<10} {"mean_iou":<10} {"notes"}')
print('-' * 75)
for r in rows:
    va = f"{r['best_val_accuracy']:.4f}" if r['best_val_accuracy'] != '' else ''
    ta = f"{r['test_accuracy']:.4f}" if r['test_accuracy'] != '' else ''
    mi = f"{r['mean_iou']:.4f}" if r['mean_iou'] != '' else ''
    print(f"{r['experiment_id']:<5} {r['task']:<16} {va:<10} {ta:<10} {mi:<10} {r['notes']}")

Saved: artifacts/runs.csv

=== Сводка ===
ID    task             val_acc    test_acc   mean_iou   notes
---------------------------------------------------------------------------
C1    classification   0.1481                           baseline CNN
C2    classification   0.1412                           with augmentations
C3    classification   0.3719                           backbone frozen
C4    classification   0.4925     0.4852                partial finetune
V1    segmentation                           0.6893     basic postprocess (threshold=0.5)
V2    segmentation                           0.6639     morpho postprocess (threshold=0.4 + close/open)
